# Flights 2013 — Build, profile and clean

NYC flights 2013 (`nycflights13`). One notebook for the whole pipeline:

| Part | Sections | What it does | Output |
|---|---|---|---|
| Setup | §0 | Download the dataset into `data/raw/` (skipped if the files are already there) | `data/raw/flights.csv`, `data/raw/airlines.csv` |
| A. Build the working table | §1–§5 | Load raw data, join airline names on `carrier`, save, verify | `data/processed/flights_working.csv` (20 columns) |
| B. D2 profile and cleaning | §6–§8 | Data profile, cleaning log (12 decisions), save the cleaned table | `data/processed/flights_clean.parquet` (24 columns) |

Section numbers §6 and §7 match the D2 report. `data/raw/` is read-only and is never edited.
Run from the project root (the folder that contains `data/`).

## 0. Download the dataset into `data/raw/`

The data comes from the [`nycflights13`](https://pypi.org/project/nycflights13/) Python package, a copy of the R package of the same name. It is installed here if it is missing.

`data/raw/` is read-only: if both CSV files already exist, nothing is downloaded or overwritten. Set `FORCE_DOWNLOAD = True` only when you deliberately want to rebuild the raw files.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

FORCE_DOWNLOAD = False

RAW_DIR = Path.cwd() / "data" / "raw"
raw_files = {"flights": RAW_DIR / "flights.csv", "airlines": RAW_DIR / "airlines.csv"}

if not FORCE_DOWNLOAD and all(p.is_file() for p in raw_files.values()):
    print("Raw files already exist; download skipped.")
else:
    if importlib.util.find_spec("nycflights13") is None:
        print("Installing nycflights13 ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "nycflights13"])
    import nycflights13

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for table, path in raw_files.items():
        data = getattr(nycflights13, table)
        data.to_csv(path, index=False, lineterminator="\n")
        print(f"Saved {path.relative_to(Path.cwd())}: {len(data):,} rows × {data.shape[1]} columns")

for table, path in raw_files.items():
    print(f"{path.name}: {path.stat().st_size / 1024:,.1f} KB")

# Part A — Build the working table

## 1. Set up libraries and file paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd()
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

FLIGHTS_PATH = RAW_DIR / "flights.csv"
AIRLINES_PATH = RAW_DIR / "airlines.csv"
WORKING_PATH = PROCESSED_DIR / "flights_working.csv"    # Part A output
CLEAN_PATH = PROCESSED_DIR / "flights_clean.parquet"    # Part B output (keeps Int64, bool and timezone types)

print(f"Project directory: {ROOT_DIR}")
print(f"Found flights.csv: {FLIGHTS_PATH.is_file()}")
print(f"Found airlines.csv: {AIRLINES_PATH.is_file()}")

## 2. Load and preview the input data

In [ ]:
if not FLIGHTS_PATH.is_file() or not AIRLINES_PATH.is_file():
    raise FileNotFoundError("Input CSV files were not found in data/raw.")

flights = pd.read_csv(FLIGHTS_PATH)
airlines = pd.read_csv(AIRLINES_PATH)

print(f"flights: {flights.shape[0]:,} rows, {flights.shape[1]} columns")
print(f"airlines: {airlines.shape[0]:,} rows, {airlines.shape[1]} columns")
display(flights.head(3))
display(airlines.head())

## 3. Join airline names to each flight

Use a `left` join to keep every flight. Carrier codes are stripped of surrounding whitespace before joining; each airline carrier code must be unique, and every flight must match an airline.

In [ ]:
if "carrier" not in flights.columns:
    raise ValueError("Flights data must contain a carrier column.")
if not {"carrier", "name"}.issubset(airlines.columns):
    raise ValueError("Airlines data must contain carrier and name columns.")
if "name" in flights.columns:
    raise ValueError("Flights data already contains a name column; check the input data.")

flights = flights.copy()
airlines = airlines[["carrier", "name"]].copy()
flights["carrier"] = flights["carrier"].astype("string").str.strip()
airlines["carrier"] = airlines["carrier"].astype("string").str.strip()
airlines["name"] = airlines["name"].astype("string").str.strip()

if flights["carrier"].isna().any() or flights["carrier"].eq("").any():
    raise ValueError("Flights data contains a missing or blank carrier code.")
if airlines[["carrier", "name"]].isna().any().any() or airlines[["carrier", "name"]].eq("").any().any():
    raise ValueError("Airlines data contains a missing or blank carrier code or airline name.")
if airlines["carrier"].duplicated().any():
    raise ValueError("Airlines data contains duplicate carrier codes.")

working_table = flights.merge(
    airlines,
    on="carrier",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched = sorted(working_table.loc[working_table["_merge"] == "left_only", "carrier"].unique())
if unmatched:
    raise ValueError(f"No airline found for these carrier codes: {unmatched}")
working_table = working_table.drop(columns="_merge")

assert len(working_table) == len(flights), "The join changed the number of flight rows."
print(f"Join successful: {len(working_table):,} flights, {len(working_table.columns)} columns.")
display(working_table[["carrier", "flight", "name"]].head())

## 4. Save the working table

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
working_table.to_csv(WORKING_PATH, index=False, lineterminator="\n")

print(f"Saved: {WORKING_PATH.relative_to(ROOT_DIR)}")
print(f"Size: {len(working_table):,} rows × {len(working_table.columns)} columns")

## 5. Reload and verify the working table

Check that the saved file contains every flight, has no missing airline names, and maps each carrier to the correct airline.

In [ ]:
saved_table = pd.read_csv(WORKING_PATH)
expected_names = airlines.set_index("carrier")["name"]
mapped_names = saved_table["carrier"].astype("string").str.strip().map(expected_names)

assert len(saved_table) == len(flights), "The output file does not contain every flight row."
assert saved_table["name"].notna().all(), "Some flights are missing an airline name."
assert saved_table["name"].astype("string").equals(mapped_names), "Airline names in the output file do not match the lookup table."

print(f"Verification successful: {len(saved_table):,} rows; all airline names match.")
display(saved_table[["carrier", "flight", "name"]].head())

# Part B — D2 profile and cleaning log

Part B starts from the **saved file**, not from `working_table` in memory, so the profile describes exactly what is on disk and Part B can be re-run on its own (run §1 first for the paths).

In [ ]:
df = pd.read_csv(WORKING_PATH)      # working copy (flights merged with airlines); raw files stay untouched
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")

## 6. Data profile (as loaded, before any cleaning)

In [ ]:
def profile(d):
    n = len(d)
    out = []
    for c in d.columns:
        s = d[c]
        nn = int(s.notna().sum())
        if pd.api.types.is_numeric_dtype(s):
            rng = f"{s.min():,.0f} to {s.max():,.0f}"
        else:
            rng = f"{s.nunique():,} distinct"
        out.append((c, str(s.dtype), nn, n - nn, round((n - nn) / n * 100, 2), rng))
    return pd.DataFrame(out, columns=["Column", "Type", "Non-null", "Missing", "Missing %", "Range or distinct values"])

profile(df)

## 7. Cleaning log
### Decision 1 — Merge check (airlines → flights on `carrier`)

In [ ]:
print("carrier->name is 1:1:", df.groupby("carrier")["name"].nunique().max() == 1 and df.groupby("name")["carrier"].nunique().max() == 1)
print("Rows with no airline name:", df["name"].isna().sum())
print("Distinct carriers:", df["carrier"].nunique())

### Decision 2 — Whole-number columns stored as float → nullable Int64

In [ ]:
int_cols = ["dep_time", "dep_delay", "arr_time", "arr_delay", "air_time"]
assert all((df[c].dropna() % 1 == 0).all() for c in int_cols)   # nothing is lost by the conversion
df[int_cols] = df[int_cols].astype("Int64")
print(df[int_cols].dtypes)

### Decision 3 — `time_hour`: text (UTC) → timezone-aware datetime in America/New_York

In [ ]:
t = pd.to_datetime(df["time_hour"], utc=True).dt.tz_convert("America/New_York")
print("Unparseable:", t.isna().sum())
agree = ((t.dt.year == df["year"]) & (t.dt.month == df["month"]) & (t.dt.day == df["day"])).sum()
print("Local date agrees with year/month/day:", f"{agree:,} of {len(df):,}")
df["time_hour"] = t

### Decision 4 — Clock times: `2400` means midnight → recode to `0` in new columns

In [ ]:
for c in ["dep_time", "arr_time"]:
    print(c, "== 2400:", int((df[c] == 2400).sum()))
    df[c + "_clock"] = df[c].mask(df[c] == 2400, 0)
print("Rows touched:", int(((df["dep_time"] == 2400) | (df["arr_time"] == 2400)).sum()))
print("sched_dep_time / sched_arr_time == 2400:", int((df["sched_dep_time"] == 2400).sum()), int((df["sched_arr_time"] == 2400).sum()))

### Decision 5 — Cancelled flights (no departure time): flag, do not drop

In [ ]:
df["cancelled"] = df["dep_time"].isna()
print("Cancelled:", int(df["cancelled"].sum()), f"({df['cancelled'].mean():.2%})")
print("All delay/air-time values missing for these rows:",
      bool(df.loc[df["cancelled"], ["dep_delay", "arr_time", "arr_delay", "air_time"]].isna().all().all()))

### Decision 6 — Missing `tailnum`: leave as missing

In [ ]:
miss_tail = df["tailnum"].isna()
print("Missing tailnum:", int(miss_tail.sum()), f"({miss_tail.mean():.2%})")
print("...of which cancelled:", int((miss_tail & df["cancelled"]).sum()))
print("...of which flew:", int((miss_tail & ~df["cancelled"]).sum()))

### Decision 7 — Flew but no arrival delay / air time (probably diverted): flag, do not drop

In [ ]:
df["no_arrival_delay"] = df["arr_delay"].isna() & ~df["cancelled"]
print("Flagged:", int(df["no_arrival_delay"].sum()), f"({df['no_arrival_delay'].mean():.2%})")
print("  has arr_time but no arr_delay:", int((df["no_arrival_delay"] & df["arr_time"].notna()).sum()))
print("  has no arr_time:", int((df["no_arrival_delay"] & df["arr_time"].isna()).sum()))
print("Reconciliation of arr_delay missing:", int(df["arr_delay"].isna().sum()), "=",
      int(df["cancelled"].sum()), "+", int(df["no_arrival_delay"].sum()))

### Decision 8 — Extreme and negative delays: keep

In [ ]:
dd = df["dep_delay"].astype("float")
print("dep_delay >= 180 min:", int((dd >= 180).sum()), "   >= 720 min:", int((dd >= 720).sum()), "   max:", dd.max())
# is the delay consistent with the clock times (modulo 24 h)? if so it is real, not a typing error
to_min = lambda x: (x // 100) * 60 + x % 100
gap = ((to_min(df["dep_time"].astype("float")) - to_min(df["sched_dep_time"]) - dd) % 1440).dropna()
print("dep_delay inconsistent with clock times:", int((gap != 0).sum()))
print("Early departures (dep_delay < 0):", int((dd < 0).sum()), "   early arrivals:", int((df["arr_delay"].astype("float") < 0).sum()))

### Decision 9 — Duplicates

In [ ]:
key = ["time_hour", "carrier", "flight", "origin", "dest", "sched_dep_time"]
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicates on flight key:", int(df.duplicated(key).sum()))

### Decision 10 — Same route, different distance

In [ ]:
g = df.groupby(["origin", "dest"])["distance"]
bad = g.nunique()[lambda s: s > 1].index
print("Routes with more than one distance:", len(bad))
mode = g.transform(lambda s: s.mode().iloc[0])
off = (df["distance"] != mode)
print("Rows that differ from the route's usual distance:", int(off.sum()), "  max gap (miles):", int((df["distance"] - mode).abs().max()))
print(df[df.set_index(["origin", "dest"]).index.isin(bad)].groupby(["origin", "dest"])["distance"].value_counts())

### Decision 11 — Implausible ground speed (distance / air_time)

In [ ]:
mph = df["distance"] / (df["air_time"].astype("float") / 60)
print("Ground speed > 650 mph:", int((mph > 650).sum()), "   max:", round(mph.max(), 1))
print(df.loc[mph > 650, ["origin", "dest", "distance", "air_time"]])
print("Ground speed < 150 mph:", int((mph < 150).sum()), "(all short hops?)  longest distance among them:", int(df.loc[mph < 150, "distance"].max()))

### Decision 12 — Categorical codes

In [ ]:
for c in ["carrier", "origin", "dest", "tailnum", "name"]:
    s = df[c].dropna()
    print(c, "| distinct:", s.nunique(), "| leading/trailing spaces:", int((s != s.str.strip()).sum()),
          "| not upper-case:", int((s != s.str.upper()).sum()) if c != "name" else "n/a")
print("Constant columns:", [c for c in df.columns if df[c].nunique(dropna=False) == 1])

### Result

In [ ]:
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   (working file had 20; +4 derived/flag columns)")
analysis_dep = df[~df["cancelled"]]                         # use for dep_delay questions
analysis_arr = df[~df["cancelled"] & ~df["no_arrival_delay"]]   # use for arr_delay / air_time questions
print("dep_delay analysis rows:", f"{len(analysis_dep):,}", "  arr_delay analysis rows:", f"{len(analysis_arr):,}")

## 8. Save and verify the cleaned table

Saved as Parquet rather than CSV: CSV would turn `Int64` back into float, the flags into text, and lose the New York timezone on `time_hour`. The two analysis subsets are not saved separately; rebuild them from the `cancelled` and `no_arrival_delay` flags.

In [ ]:
df.to_parquet(CLEAN_PATH, index=False)

clean_check = pd.read_parquet(CLEAN_PATH)
assert clean_check.shape == df.shape, "Saved file has a different shape."
assert (clean_check.dtypes == df.dtypes).all(), "Column types changed on save."
assert len(clean_check) == len(flights), "Cleaned table does not contain every flight."
pd.testing.assert_frame_equal(clean_check, df)

print(f"Saved: {CLEAN_PATH.relative_to(ROOT_DIR)}")
print(f"Size: {len(clean_check):,} rows × {clean_check.shape[1]} columns; reload matches exactly.")